In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

True Tesla T4


In [2]:
import pandas as pd, os

REAL = '/kaggle/input/datasets/akshatyuvan/musiccaps-real-16k'
SYN  = '/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw'

# Reuse M1's exact subset rather than re-sampling. The comparison table is
# only valid if both models see identical clips in identical splits.
sub = pd.read_csv('/kaggle/input/datasets/akshatyuvan/fmc-m1-features/subset.csv')
sub['path'] = (sub.path
    .str.replace('/kaggle/working/real', REAL, regex=False)
    .str.replace('/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw', SYN, regex=False))

print(sub.shape)
print(sub.groupby(['split','label']).size())
print('paths ok:', sub.path.head(50).map(os.path.exists).all())

(10697, 5)
split  label
test   0         804
       1         800
train  0        3746
       1        3745
val    0         802
       1         800
dtype: int64
paths ok: True


In [3]:
import numpy as np, soundfile as sf, librosa
from joblib import Parallel, delayed

SR, N_MELS, N_FFT, HOP = 16000, 128, 1024, 160
N_SAMPLES = 160000          # exactly 10.0 s; crops the 10.01 s real clips so
                            # duration cannot act as a shortcut
N_FRAMES  = N_SAMPLES // HOP + 1    # 1001

def mel(path):
    y, _ = sf.read(path, dtype='float32')
    y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                       hop_length=HOP, n_mels=N_MELS)
    S = librosa.power_to_db(S, ref=np.max)
    return S[:, :N_FRAMES].astype(np.float16)   # float16 halves disk and RAM

M = np.stack(Parallel(n_jobs=4, verbose=5)(delayed(mel)(p) for p in sub.path))
labels = sub.label.values.astype(np.int64)
print(M.shape, M.dtype, 'NaNs:', np.isnan(M).sum())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   25.8s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   26.8s
[Parallel(n_jobs=4)]: Done 480 tasks      | elapsed:   30.0s
[Parallel(n_jobs=4)]: Done 984 tasks      | elapsed:   34.6s
[Parallel(n_jobs=4)]: Done 1632 tasks      | elapsed:   40.3s
[Parallel(n_jobs=4)]: Done 2424 tasks      | elapsed:   47.4s
[Parallel(n_jobs=4)]: Done 3360 tasks      | elapsed:   55.8s
[Parallel(n_jobs=4)]: Done 4440 tasks      | elapsed:  1.1min
[Parallel(n_jobs=4)]: Done 5664 tasks      | elapsed:  1.3min
[Parallel(n_jobs=4)]: Done 7032 tasks      | elapsed:  1.5min
[Parallel(n_jobs=4)]: Done 8544 tasks      | elapsed:  1.7min
[Parallel(n_jobs=4)]: Done 10200 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done 10697 out of 10697 | elapsed:  2.1min finished


(10697, 128, 1001) float16 NaNs: 0


In [4]:
import os, json, numpy as np
os.makedirs('/kaggle/working/m2_mels', exist_ok=True)
np.save('/kaggle/working/m2_mels/mels.npy', M)
np.save('/kaggle/working/m2_mels/labels.npy', labels)
sub.to_csv('/kaggle/working/m2_mels/subset.csv', index=False)
json.dump({"title":"fmc-m2-mels","id":"akshatyuvan/fmc-m2-mels",
           "licenses":[{"name":"CC0-1.0"}]},
          open('/kaggle/working/m2_mels/dataset-metadata.json','w'))
!ls -lh /kaggle/working/m2_mels

total 2.6G
-rw-r--r-- 1 root root   92 Oct  4 09:26 dataset-metadata.json
-rw-r--r-- 1 root root  84K Oct  4 09:26 labels.npy
-rw-r--r-- 1 root root 2.6G Oct  4 09:26 mels.npy
-rw-r--r-- 1 root root 1.1M Oct  4 09:26 subset.csv


In [6]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret("kaggle_token")
!kaggle datasets create -p /kaggle/working/m2_mels

Starting upload for file mels.npy
100%|██████████████████████████████████████| 2.55G/2.55G [00:37<00:00, 72.3MB/s]
Upload successful: mels.npy (3GB)
Starting upload for file subset.csv
100%|██████████████████████████████████████| 1.06M/1.06M [00:00<00:00, 1.67MB/s]
Upload successful: subset.csv (1MB)
Starting upload for file labels.npy
100%|███████████████████████████████████████| 83.7k/83.7k [00:00<00:00, 134kB/s]
Upload successful: labels.npy (84KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/akshatyuvan/fmc-m2-mels


In [7]:
!kaggle datasets list -m

ref                             title                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------  ------------------  ----------  --------------------------  -------------  ---------  ---------------  
akshatyuvan/tweets              tweets                 6791606  2026-04-14 17:51:09.203000              0          0  0.11764706       
akshatyuvan/fakemusiccaps-raw   fakemusiccaps-raw            0  2026-10-03 13:54:08.600000              0          0  0.25             
akshatyuvan/fmc-manifest        fmc-manifest            401901  2026-10-03 16:02:51.093000              0          0  0.23529412       
akshatyuvan/musiccaps-real-16k  musiccaps-real-16k  3158513578  2026-10-03 16:01:02.200000              0          0  0.25             
akshatyuvan/fmc-m1-features     fmc-m1-features        5121449  2026-10-03 21:14:47.177000              0          0  0.23529412       


In [8]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

idx = {s: np.where((sub.split == s).values)[0] for s in ['train','val','test']}

# Normalisation statistics come from the training split only.
# Computing them over all data would leak test information.
tr_mean = float(M[idx['train']].astype(np.float32).mean())
tr_std  = float(M[idx['train']].astype(np.float32).std())
print('train mel mean/std:', round(tr_mean,3), round(tr_std,3))

class MelDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        j = self.rows[i]
        x = (M[j].astype(np.float32) - tr_mean) / tr_std
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(float(labels[j]))

loaders = {
    s: DataLoader(MelDS(idx[s]), batch_size=32, shuffle=(s=='train'),
                  num_workers=2, pin_memory=True)
    for s in ['train','val','test']
}
xb, yb = next(iter(loaders['train']))
print('batch:', xb.shape, yb.shape)

train mel mean/std: -45.663 17.12
batch: torch.Size([32, 1, 128, 1001]) torch.Size([32])


In [9]:
import torch.nn as nn

class M2CNN(nn.Module):
    """Four conv blocks over the log-mel plane, then global pooling.

    Global average pooling rather than flatten keeps the model agnostic to
    clip length and, more importantly for this project, preserves a spatial
    feature map that Grad-CAM can be computed on at the final stage.
    """
    def __init__(self):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(blk(1,32), blk(32,64), blk(64,128), blk(128,256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256,1))
    def forward(self, x): return self.head(self.features(x))

dev = 'cuda'
model = M2CNN().to(dev)
print('params:', sum(p.numel() for p in model.parameters()))

params: 1173857


In [10]:
import time, copy

opt    = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
lossf  = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda')   # T4 is Turing: fp16, not bf16

best_val, best_state, patience, bad = 0.0, None, 5, 0
t0 = time.time()

for ep in range(30):
    model.train()
    for xb, yb in loaders['train']:
        xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            loss = lossf(model(xb).squeeze(1), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in loaders['val']:
            xb, yb = xb.to(dev), yb.to(dev)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                p = (model(xb).squeeze(1) > 0).float()
            correct += (p == yb).sum().item(); total += len(yb)
    va = correct / total
    print(f'epoch {ep:2d}  val acc {va:.4f}  ({time.time()-t0:.0f}s)', flush=True)

    if va > best_val:
        best_val, bad = va, 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, '/kaggle/working/m2_best.pt')   # survives a crash
    else:
        bad += 1
        if bad >= patience:
            print('early stop at epoch', ep); break

model.load_state_dict(best_state)
m2_time = time.time() - t0
print(f'best val acc {best_val:.4f} | train time {m2_time:.1f}s')

epoch  0  val acc 0.6111  (58s)
epoch  1  val acc 0.8914  (121s)
epoch  2  val acc 0.8901  (188s)
epoch  3  val acc 0.8071  (255s)
epoch  4  val acc 0.8315  (322s)
epoch  5  val acc 0.8845  (388s)
epoch  6  val acc 0.9588  (454s)
epoch  7  val acc 0.9132  (521s)
epoch  8  val acc 0.9613  (587s)
epoch  9  val acc 0.9700  (653s)
epoch 10  val acc 0.8246  (720s)
epoch 11  val acc 0.8589  (786s)
epoch 12  val acc 0.9519  (852s)
epoch 13  val acc 0.9544  (918s)
epoch 14  val acc 0.9694  (985s)
early stop at epoch 14
best val acc 0.9700 | train time 984.6s


In [11]:
import numpy as np
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

model.eval(); scores, trues = [], []
with torch.no_grad():
    for xb, yb in loaders['test']:
        xb = xb.to(dev)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            s = torch.sigmoid(model(xb).squeeze(1).float())
        scores.append(s.cpu().numpy()); trues.append(yb.numpy())

score = np.concatenate(scores); true = np.concatenate(trues)
pred  = (score > 0.5).astype(int)

fpr, tpr, _ = roc_curve(true, score)
eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])

m2 = {"balanced_accuracy": balanced_accuracy_score(true, pred),
      "precision": precision_score(true, pred),
      "recall":    recall_score(true, pred),
      "f1":        f1_score(true, pred),
      "auc":       roc_auc_score(true, score),
      "eer":       eer}
for k, v in m2.items(): print(f'{k:20s} {v:.4f}')
print(f'\ntrain time {m2_time:.1f}s | seed {SEED} | device {dev} | params {sum(p.numel() for p in model.parameters())}')

balanced_accuracy    0.9732
precision            0.9834
recall               0.9625
f1                   0.9728
auc                  0.9961
eer                  0.0249

train time 984.6s | seed 42 | device cuda | params 1173857


In [12]:
import json
json.dump({"model":"M2_CNN","seed":SEED,"device":"cuda",
           "train_time_s":round(m2_time,1),"best_val_acc":round(best_val,4),
           "params":sum(p.numel() for p in model.parameters()),
           **{k:round(v,4) for k,v in m2.items()}},
          open('/kaggle/working/m2_mels/m2_results.json','w'), indent=2)
import shutil; shutil.copy('/kaggle/working/m2_best.pt', '/kaggle/working/m2_mels/')
!kaggle datasets version -p /kaggle/working/m2_mels -m "add m2 results and checkpoint"

Starting upload for file m2_best.pt
100%|██████████████████████████████████████| 4.50M/4.50M [00:00<00:00, 5.99MB/s]
Upload successful: m2_best.pt (5MB)
Starting upload for file m2_results.json
100%|████████████████████████████████████████████| 251/251 [00:00<00:00, 407B/s]
Upload successful: m2_results.json (251B)
Starting upload for file mels.npy
100%|██████████████████████████████████████| 2.55G/2.55G [00:39<00:00, 69.9MB/s]
Upload successful: mels.npy (3GB)
Starting upload for file subset.csv
100%|██████████████████████████████████████| 1.06M/1.06M [00:00<00:00, 1.64MB/s]
Upload successful: subset.csv (1MB)
Starting upload for file labels.npy
100%|███████████████████████████████████████| 83.7k/83.7k [00:00<00:00, 132kB/s]
Upload successful: labels.npy (84KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/akshatyuvan/fmc-m2-mels
